# 10 — Direction 1, step A: is the thermal core the linear system we think?

`hvac_controller/d1/core.py` wraps the simulator's `FiniteDifferenceSolver.solve` during a baseline (`pid_water`)
rollout and, at every step:

- reduces the heat input to one number per zone and checks it rebuilds the field (`max_q_rebuild_err_w`);
- re-runs the step with our own pure function from the simulator's state (`max_one_step_err_k`: must be ~0);
- advances a replay that never sees the intra-room mixing swap (`free_run_*_err_k`: what the swap changes in the
  temperatures the VAVs sense; ~0 when the swap is switched off).

Three scenarios (mild, hot, cold; 30–85 zones), each with the swap off (`p = 0`) and on (as shipped), 4 days each.
The swap-on runs also save the matrices and logged inputs that step B (reduced model) replays.

**Setup:** Internet on; attach the competition data. CPU. Expected ≈ 25–40 min.
**Pass:** one-step error ≤ 1e-9 K everywhere; with the swap off, free-run error ≤ 1e-6 K; with it on, free-run error
< 0.05 K after 24 h. Output: `/kaggle/working/d1_core.zip`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
SCENARIOS = ["building_train_1/atlanta_2", "building_train_2/phoenix_6", "building_train_8/fairbanks_8"]
N_STEPS = 1152  # 4 days of 5-minute steps

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)

from hvac_controller.d1.core import run_core_check

In [ ]:
OUT = Path("/kaggle/working/d1_core")
reports = []
for scenario in SCENARIOS:
    cfg = REPO / "smart_buildings" / "training_scenarios" / scenario / "config.gin"
    for swap in (False, True):
        tag = scenario.replace("/", "__") + f"__swap{int(swap)}"
        print(f"[core] {tag} ...", flush=True)
        try:
            r = run_core_check(cfg, N_STEPS, swap, out_dir=OUT / tag if swap else None)
        except Exception as e:  # keep going; the report says what failed
            import traceback
            r = {"config": str(cfg), "swap": swap, "error": f"{type(e).__name__}: {e}",
                 "traceback": traceback.format_exc()}
        r["tag"] = tag
        reports.append(r)
        print(f"[core] {tag} done in {r.get('seconds', 0):.0f}s", flush=True)

OUT.mkdir(parents=True, exist_ok=True)
(OUT / "reports.json").write_text(json.dumps({"commit": COMMIT, "reports": reports}, indent=1))
shutil.make_archive(str(OUT), "zip", OUT)
print("wrote", f"{OUT}.zip")

In [ ]:
keys = ["steps", "n_solved", "n_zones", "n_vavs", "h", "max_one_step_err_k", "max_q_rebuild_err_w",
        "free_run_max_err_k", "free_run_mean_err_k", "share_steps_air_on", "seconds"]
for r in reports:
    if "error" in r:
        print(r["tag"], "ERROR", r["error"])
    else:
        print(r["tag"], {k: r.get(k) for k in keys})